In [1]:
import pandas as pd
import numpy as np

In [2]:
%pip install sqlalchemy pymysql pandas cryptography

Note: you may need to restart the kernel to use updated packages.


In [3]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url = URL.create(
    "mysql+pymysql",
    username="root",
    password="NewPass123",
    host="localhost",
    port=3306,
)
root_engine = create_engine(url)

with root_engine.connect() as conn:
    print(conn.execute(text("SELECT VERSION()")).scalar())

8.0.31


In [4]:
with root_engine.connect() as conn:
    conn.execute(text(
        "CREATE DATABASE IF NOT EXISTS nba_db "
        "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"))
    conn.commit()
engine = create_engine(
    URL.create("mysql+pymysql", username="root", password="NewPass123",
               host="localhost", port=3306, database="nba_db",
               query={"charset": "utf8mb4"}))
with engine.connect() as conn:
    print(conn.execute(text("SELECT DATABASE()")).scalar())

nba_db


In [10]:
from typing import Optional
from sqlalchemy import (String, Integer, Float, Boolean, Date, ForeignKey,UniqueConstraint)
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

In [25]:

class Base(DeclarativeBase):
    pass
class Player(Base):
    __tablename__ = "player"
    player_id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(100))
    birth_date: Mapped[Optional[Date]] = mapped_column(Date)
    height_cm: Mapped[Optional[float]] = mapped_column(Float)
    weight_kg: Mapped[Optional[float]] = mapped_column(Float)

class College(Base):
    __tablename__ = "college"
    college_id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(120), unique=True)

class PlayerCollege(Base):
    __tablename__ = "player_college"
    player_id: Mapped[int] = mapped_column(ForeignKey("player.player_id"), primary_key=True)
    college_id: Mapped[int] = mapped_column(ForeignKey("college.college_id"), primary_key=True)

class Team(Base):
    __tablename__ = "team"
    team_id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    abbr:Mapped[str] = mapped_column(String(5), unique=True)
    name:Mapped[str] = mapped_column(String(60))

class Season(Base):
    __tablename__ = "season"
    season_year: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=False)
    label: Mapped[str] = mapped_column(String(7))
    champion_team_id: Mapped[Optional[int]] = mapped_column(ForeignKey("team.team_id"))

class TeamSeason(Base):
    __tablename__ = "team_season"
    team_id: Mapped[int] = mapped_column(ForeignKey("team.team_id"), primary_key=True)
    season_year: Mapped[int] = mapped_column(ForeignKey("season.season_year"), primary_key=True)
    wins: Mapped[Optional[int]] = mapped_column(Integer)
    losses: Mapped[Optional[int]] = mapped_column(Integer)
    srs: Mapped[Optional[float]] = mapped_column(Float)
    pace: Mapped[Optional[float]] = mapped_column(Float)
    ortg: Mapped[Optional[float]] = mapped_column(Float)
    drtg: Mapped[Optional[float]] = mapped_column(Float)
    playoffs: Mapped[Optional[str]] = mapped_column(String(40))

class PlayerSeason(Base):
    __tablename__ = "player_season"
    player_season_id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    player_id: Mapped[int] = mapped_column(ForeignKey("player.player_id"))
    team_id: Mapped[Optional[int]] = mapped_column(ForeignKey("team.team_id"))
    season_year: Mapped[int] = mapped_column(ForeignKey("season.season_year"))
    pos: Mapped[Optional[str]] = mapped_column(String(5))
    exp: Mapped[Optional[int]] = mapped_column(Integer)
    salary: Mapped[Optional[int]] = mapped_column(Integer)
    is_total: Mapped[bool] = mapped_column(Boolean, default=False)
    __table_args__ = (UniqueConstraint("player_id","season_year","team_id","is_total",name="uq_player_season_team_total"),)

class SeasonTotals(Base):
    __tablename__ = "season_totals"
    player_season_id: Mapped[int] = mapped_column(ForeignKey("player_season.player_season_id"), primary_key=True)
    g: Mapped[Optional[int]] = mapped_column(Integer)
    gs: Mapped[Optional[int]] = mapped_column(Integer)
    mp: Mapped[Optional[int]] = mapped_column(Integer)
    fg: Mapped[Optional[int]] = mapped_column(Integer)
    fga: Mapped[Optional[int]] = mapped_column(Integer)
    fg3: Mapped[Optional[int]] = mapped_column(Integer)
    fg3a: Mapped[Optional[int]] = mapped_column(Integer)
    fg2: Mapped[Optional[int]] = mapped_column(Integer)
    fg2a: Mapped[Optional[int]] = mapped_column(Integer)
    ft: Mapped[Optional[int]] = mapped_column(Integer)
    fta: Mapped[Optional[int]] = mapped_column(Integer)
    orb: Mapped[Optional[int]] = mapped_column(Integer)
    drb: Mapped[Optional[int]] = mapped_column(Integer)
    trb: Mapped[Optional[int]] = mapped_column(Integer)
    ast: Mapped[Optional[int]] = mapped_column(Integer)
    stl: Mapped[Optional[int]] = mapped_column(Integer)
    blk: Mapped[Optional[int]] = mapped_column(Integer)
    tov: Mapped[Optional[int]] = mapped_column(Integer)
    pf: Mapped[Optional[int]] = mapped_column(Integer)
    pts: Mapped[Optional[int]] = mapped_column(Integer)
    triple_doubles: Mapped[Optional[int]] = mapped_column(Integer)

class SeasonAdvanced(Base):
    __tablename__ = "season_advanced"
    player_season_id: Mapped[int] = mapped_column(ForeignKey("player_season.player_season_id"), primary_key=True)
    per: Mapped[Optional[float]] = mapped_column(Float)
    ts_pct: Mapped[Optional[float]] = mapped_column(Float)
    fg3a_rate: Mapped[Optional[float]] = mapped_column(Float)
    ft_rate: Mapped[Optional[float]] = mapped_column(Float)
    orb_pct: Mapped[Optional[float]] = mapped_column(Float)
    drb_pct: Mapped[Optional[float]] = mapped_column(Float)
    trb_pct: Mapped[Optional[float]] = mapped_column(Float)
    ast_pct: Mapped[Optional[float]] = mapped_column(Float)
    stl_pct: Mapped[Optional[float]] = mapped_column(Float)
    blk_pct: Mapped[Optional[float]] = mapped_column(Float)
    tov_pct: Mapped[Optional[float]] = mapped_column(Float)
    usg_pct: Mapped[Optional[float]] = mapped_column(Float)
    ows: Mapped[Optional[float]] = mapped_column(Float)
    dws: Mapped[Optional[float]] = mapped_column(Float)
    ws: Mapped[Optional[float]] = mapped_column(Float)
    ws_per_48: Mapped[Optional[float]] = mapped_column(Float)
    obpm: Mapped[Optional[float]] = mapped_column(Float)
    dbpm: Mapped[Optional[float]] = mapped_column(Float)
    bpm: Mapped[Optional[float]] = mapped_column(Float)
    vorp: Mapped[Optional[float]] = mapped_column(Float)

class MvpVote(Base):
    __tablename__ = "mvp_vote"
    season_year: Mapped[int] = mapped_column(ForeignKey("season.season_year"), primary_key=True)
    player_id: Mapped[int] = mapped_column(ForeignKey("player.player_id"), primary_key=True)
    rank_num: Mapped[Optional[int]] = mapped_column(Integer)
    is_tied: Mapped[bool] = mapped_column(Boolean, default=False)
    first_votes: Mapped[Optional[int]] = mapped_column(Integer)
    pts_won: Mapped[Optional[int]] = mapped_column(Integer)
    pts_max: Mapped[Optional[int]] = mapped_column(Integer)
    share: Mapped[Optional[float]] = mapped_column(Float)

class PlayerSeasonAward(Base):
    __tablename__ = "player_season_award"
    player_season_id: Mapped[int] = mapped_column(ForeignKey("player_season.player_season_id"), primary_key=True)
    award_code: Mapped[str] = mapped_column(String(15), primary_key=True)
Base.metadata.create_all(engine)
print(sorted(Base.metadata.tables))

['college', 'mvp_vote', 'player', 'player_college', 'player_season', 'player_season_award', 'season', 'season_advanced', 'season_totals', 'team', 'team_season']


In [263]:
from sqlalchemy import text

with engine.begin() as conn:
    for t in ["player_season_award", "season_advanced", "season_totals", "mvp_vote",
              "player_season", "player_college", "college", "team_season",
              "season", "player", "team"]:
        conn.execute(text(f"DELETE FROM {t}"))

In [267]:
from pathlib import Path
import pandas as pd

DATA = Path(r"C:\Users\Snapp\Desktop\Report\Down\NBA")

players  = pd.read_csv(DATA / "players.csv")
rosters  = pd.read_csv(DATA / "rosters.csv")
salaries = pd.read_csv(DATA / "rosters_salaries.csv")
totals   = pd.read_csv(DATA / "totals_pss.csv")
advanced = pd.read_csv(DATA / "advanced_pss.csv")
mvp      = pd.read_csv(DATA / "micheal_jordan_trophy.csv")
teams_s  = pd.read_csv(DATA / "teams_all_season.csv")
seasons  = pd.read_csv(DATA / "seasons.csv")

# ردیف‌های خالی و ردیف League Average در آمار
totals   = totals[totals["Player"].notna() & (totals["Player"] != "League Average")]
advanced = advanced[advanced["Player"].notna() & (advanced["Player"] != "League Average")]

# بازیکن‌ها: فقط فصل‌های ۲۰۲۰ به بعد + بازیکنانی که در players.csv نیستند
players = players[players["To"] >= 2020]
missing = rosters[~rosters["Player"].isin(players["Player"])].drop_duplicates("Player")
missing = missing.rename(columns={"College": "Colleges"})
players = pd.concat([players, missing])

h = players["Ht"].str.split("-", expand=True).astype(float)
players["height_cm"] = (h[0] * 12 + h[1]) * 2.54
players["weight_kg"] = players["Wt"] * 0.4536
players["birth_date"] = pd.to_datetime(players["Birth Date"])
players = players.rename(columns={"Player": "name"})

# تجربه و حقوق
rosters["exp"] = rosters["Exp"].replace("R", "0").astype(int)
salaries = salaries.rename(columns={"Unnamed: 3": "Player"})
salaries["salary"] = pd.to_numeric(salaries["Salary"].str.replace("[$,]", "", regex=True))
# بعضی بازیکنان در یک تیم چند قرارداد دارند (مثلاً ۱۰ روزه)؛ حقوقشان جمع زده می‌شود
salaries = salaries.groupby(["Season", "Team", "Player"], as_index=False)["salary"].sum(min_count=1)
salaries["salary"] = salaries["salary"].astype("Int64")

# رتبه‌ی MVP
mvp["rank_num"] = mvp["Rank"].str.replace("T", "").astype(int)
mvp["is_tied"] = mvp["Rank"].str.endswith("T")

# فصل‌ها: 2019-20 → 2020
seasons = seasons[seasons["Season"].str.match(r"^\d{4}-\d{2}$", na=False)].copy()
seasons["season_year"] = seasons["Season"].str[:4].astype(int) + 1
seasons = seasons[seasons["season_year"].between(2020, 2026)]

teams_s["season_year"] = teams_s["Season"].str[:4].astype(int) + 1
teams_s = teams_s[teams_s["season_year"].between(2020, 2026)].copy()
teams_s["Team"] = teams_s["Team"].str.replace("*", "", regex=False)

In [268]:
TEAMS = {
 "ATL":"Atlanta Hawks", "BOS":"Boston Celtics", "BRK":"Brooklyn Nets", "CHI":"Chicago Bulls",
 "CHO":"Charlotte Hornets", "CLE":"Cleveland Cavaliers", "DAL":"Dallas Mavericks",
 "DEN":"Denver Nuggets", "DET":"Detroit Pistons", "GSW":"Golden State Warriors",
 "HOU":"Houston Rockets", "IND":"Indiana Pacers", "LAC":"Los Angeles Clippers",
 "LAL":"Los Angeles Lakers", "MEM":"Memphis Grizzlies", "MIA":"Miami Heat",
 "MIL":"Milwaukee Bucks", "MIN":"Minnesota Timberwolves", "NOP":"New Orleans Pelicans",
 "NYK":"New York Knicks", "OKC":"Oklahoma City Thunder", "ORL":"Orlando Magic",
 "PHI":"Philadelphia 76ers", "PHO":"Phoenix Suns", "POR":"Portland Trail Blazers",
 "SAC":"Sacramento Kings", "SAS":"San Antonio Spurs", "TOR":"Toronto Raptors",
 "UTA":"Utah Jazz", "WAS":"Washington Wizards",
}

# ---------- team ----------
pd.DataFrame({"abbr": list(TEAMS), "name": list(TEAMS.values())}).to_sql("team", engine, if_exists="append", index=False)
t = pd.read_sql("SELECT team_id, abbr, name FROM team", engine)
abbr_id = dict(zip(t["abbr"], t["team_id"]))
name_id = dict(zip(t["name"], t["team_id"]))

# ---------- season ----------
seasons["champion_team_id"] = seasons["Champion"].map(name_id).astype("Int64")
seasons = seasons.rename(columns={"Season": "label"})
seasons[["season_year", "label", "champion_team_id"]].to_sql("season", engine, if_exists="append", index=False)

# ---------- team_season ----------
teams_s["team_id"] = teams_s["Team"].map(name_id)
teams_s = teams_s.rename(columns={"W": "wins", "L": "losses", "SRS": "srs", "Pace": "pace",
                                  "ORtg": "ortg", "DRtg": "drtg", "Playoffs": "playoffs"})
teams_s[["team_id", "season_year", "wins", "losses", "srs", "pace", "ortg", "drtg", "playoffs"]] \
    .to_sql("team_season", engine, if_exists="append", index=False)

# ---------- player ----------
players[["name", "birth_date", "height_cm", "weight_kg"]].to_sql("player", engine, if_exists="append", index=False)
p = pd.read_sql("SELECT player_id, name FROM player", engine)
player_id = dict(zip(p["name"], p["player_id"]))

# ---------- college و player_college ----------
pc = players[["name", "Colleges"]].dropna()
pc["college"] = pc["Colleges"].str.split(", ")
pc = pc.explode("college")
pd.DataFrame({"name": pc["college"].unique()}).to_sql("college", engine, if_exists="append", index=False)
c = pd.read_sql("SELECT college_id, name FROM college", engine)
college_id = dict(zip(c["name"], c["college_id"]))
pd.DataFrame({"player_id": pc["name"].map(player_id), "college_id": pc["college"].map(college_id)}) \
    .drop_duplicates().to_sql("player_college", engine, if_exists="append", index=False)

# ---------- player_season ----------
ps = totals[["Season", "Player", "Team", "Pos"]].copy()
ps["player_id"] = ps["Player"].map(player_id)
ps["team_id"] = ps["Team"].map(abbr_id).astype("Int64")      # برای 2TM و 3TM خالی می‌ماند
ps["is_total"] = ps["Team"].str.contains("TM")
ps["season_year"] = ps["Season"]
ps["pos"] = ps["Pos"]
ps = ps.merge(rosters[["Season", "Team", "Player", "exp"]], on=["Season", "Team", "Player"], how="left")
ps = ps.merge(salaries[["Season", "Team", "Player", "salary"]], on=["Season", "Team", "Player"], how="left")
ps[["player_id", "team_id", "season_year", "pos", "exp", "salary", "is_total"]] \
    .to_sql("player_season", engine, if_exists="append", index=False)

db = pd.read_sql("SELECT player_season_id, player_id, season_year, team_id FROM player_season", engine)
ps = ps.merge(db, on=["player_id", "season_year", "team_id"])

# ---------- season_totals ----------
tt = totals.merge(ps[["Season", "Player", "Team", "player_season_id"]], on=["Season", "Player", "Team"])
tt = tt.rename(columns={"G": "g", "GS": "gs", "MP": "mp", "FG": "fg", "FGA": "fga", "3P": "fg3", "3PA": "fg3a",
                        "2P": "fg2", "2PA": "fg2a", "FT": "ft", "FTA": "fta", "ORB": "orb", "DRB": "drb",
                        "TRB": "trb", "AST": "ast", "STL": "stl", "BLK": "blk", "TOV": "tov", "PF": "pf",
                        "PTS": "pts", "Trp-Dbl": "triple_doubles"})
cols = ["g", "gs", "mp", "fg", "fga", "fg3", "fg3a", "fg2", "fg2a", "ft", "fta", "orb", "drb",
        "trb", "ast", "stl", "blk", "tov", "pf", "pts", "triple_doubles"]
tt[cols] = tt[cols].astype("Int64")
tt[["player_season_id"] + cols].to_sql("season_totals", engine, if_exists="append", index=False)

# ---------- season_advanced ----------
aa = advanced.merge(ps[["Season", "Player", "Team", "player_season_id"]], on=["Season", "Player", "Team"])
aa = aa.rename(columns={"PER": "per", "TS%": "ts_pct", "3PAr": "fg3a_rate", "FTr": "ft_rate", "ORB%": "orb_pct",
                        "DRB%": "drb_pct", "TRB%": "trb_pct", "AST%": "ast_pct", "STL%": "stl_pct",
                        "BLK%": "blk_pct", "TOV%": "tov_pct", "USG%": "usg_pct", "OWS": "ows", "DWS": "dws",
                        "WS": "ws", "WS/48": "ws_per_48", "OBPM": "obpm", "DBPM": "dbpm", "BPM": "bpm", "VORP": "vorp"})
cols = ["per", "ts_pct", "fg3a_rate", "ft_rate", "orb_pct", "drb_pct", "trb_pct", "ast_pct", "stl_pct", "blk_pct",
        "tov_pct", "usg_pct", "ows", "dws", "ws", "ws_per_48", "obpm", "dbpm", "bpm", "vorp"]
aa[["player_season_id"] + cols].to_sql("season_advanced", engine, if_exists="append", index=False)

# ---------- player_season_award ----------
aw = tt[["player_season_id", "Awards"]].dropna()
aw["award_code"] = aw["Awards"].str.split(",")
aw = aw.explode("award_code")[["player_season_id", "award_code"]].drop_duplicates()
aw.to_sql("player_season_award", engine, if_exists="append", index=False)

# ---------- mvp_vote ----------
mvp["player_id"] = mvp["Player"].map(player_id)
mvp = mvp.rename(columns={"Season": "season_year", "First": "first_votes", "Pts Won": "pts_won",
                          "Pts Max": "pts_max", "Share": "share"})
mvp[["season_year", "player_id", "rank_num", "is_tied", "first_votes", "pts_won", "pts_max", "share"]] \
    .to_sql("mvp_vote", engine, if_exists="append", index=False)

81